# Stacking Ensemble Deneyleri (v2)

**Amac**: Temiz (clean_fe) veri uzerinde stacking ensemble ile tek modellere gore performans artisi saglamak.

| Meta-learner | Base Modeller | Aciklama |
|-------------|---------------|----------|
| LogisticRegression | LightGBM + XGBoost + SklearnMLP | Basit, yorumlanabilir meta-learner |
| LightGBM | LightGBM + XGBoost + SklearnMLP | Non-linear meta-learner |

**Stacking CV**: StratifiedKFold(5) -- base model tahminleri CV ile uretilir, stacking leakage onlenir.

In [1]:
import sys, os
import warnings
import time

import pandas as pd
import numpy as np
import joblib

# Proje kokunu path'e ekle
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

from config import *
from src.features import prepare_data
from src.stacking import STACKING_BUILDERS
from src.metrics import optimize_threshold, compute_all_metrics
from src.utils import get_train_test_data

from sklearn.metrics import f1_score as _f1_score
warnings.filterwarnings('ignore')

print(f'Proje koku: {PROJECT_ROOT}')
print(f'Stacking tipleri: {list(STACKING_BUILDERS.keys())}')

Proje koku: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model
Stacking tipleri: ['stacking_lr', 'stacking_lgbm']


In [2]:
# Veri yukle ve clean_fe hazirla
df_raw = pd.read_csv(DATA_PATH)
print(f'Veri boyutu: {df_raw.shape}')

# Sadece clean_fe modu kullanilacak (sizintisiz + gelismis FE)
df_clean_fe = prepare_data(df_raw.copy(), 'clean_fe')
print(f'Clean-FE boyutu: {df_clean_fe.shape}')
print(f'Sinif dagilimi:\n{df_clean_fe["target"].value_counts()}')

Veri boyutu: (4287, 119)
  Conservation PCA: 4 skor -> 2 PC (varyans: 97.69%)
  DNA_11mer_Ref/DNA_11mer_Alt: 16 k-mer + 16 delta
  Prot_11mer_Ref/Prot_11mer_Alt: 431 k-mer + 431 delta
  mutation_type: 12 one-hot feature
  aa_change: 150 one-hot feature
  98 sizintili/gereksiz sutun kaldirildi
  r>0.99 kopya filtreleme: 19 sutun dusuruldu
  Sifir varyans filtreleme: 13 sutun dusuruldu
[Clean-FE] Veri boyutu: 4287 x 1497
Clean-FE boyutu: (4287, 1497)
Sinif dagilimi:
target
0    2920
1    1367
Name: count, dtype: int64


In [3]:
# Stacking konfigurasyonlari olustur
STACKING_CONFIGS = []

for stack_type in STACKING_TYPES:
    # Multi-panel
    STACKING_CONFIGS.append({
        'config_id': f'{stack_type}_clean_fe_multi_panel',
        'stack_type': stack_type,
        'mode': 'multi_panel',
        'panel': None,
    })
    # Single-panel
    for panel in PANELS_SINGLE:
        STACKING_CONFIGS.append({
            'config_id': f'{stack_type}_clean_fe_single_{panel}',
            'stack_type': stack_type,
            'mode': 'single_panel',
            'panel': panel,
        })

print(f'Toplam stacking konfigurasyonu: {len(STACKING_CONFIGS)}')
for c in STACKING_CONFIGS:
    print(f'  {c["config_id"]}')

Toplam stacking konfigurasyonu: 8
  stacking_lr_clean_fe_multi_panel
  stacking_lr_clean_fe_single_General
  stacking_lr_clean_fe_single_Hereditary_Cancer
  stacking_lr_clean_fe_single_PAH
  stacking_lgbm_clean_fe_multi_panel
  stacking_lgbm_clean_fe_single_General
  stacking_lgbm_clean_fe_single_Hereditary_Cancer
  stacking_lgbm_clean_fe_single_PAH


In [4]:
def run_stacking_config(config, df_prepared):
    """Tek bir stacking konfigurasyonunu calistir."""
    config_id = config['config_id']
    stack_type = config['stack_type']
    mode = config['mode']
    panel = config.get('panel', None)

    X_train, X_test, y_train, y_test, cat_features = get_train_test_data(
        df_prepared, mode, panel
    )

    print(f'  Veri: {X_train.shape[0]} train, {X_test.shape[0]} test, {X_train.shape[1]} feature')

    builder = STACKING_BUILDERS[stack_type]
    start_time = time.time()
    model, study, best_thr, y_pred_proba = builder(
        X_train, y_train, X_test, y_test, cat_features, use_focal=True
    )
    elapsed = time.time() - start_time

    y_pred = (y_pred_proba >= best_thr).astype(int)
    metrics = compute_all_metrics(y_test, y_pred, y_pred_proba)

    y_pred_default = (y_pred_proba >= 0.5).astype(int)
    f1_default = _f1_score(y_test, y_pred_default, zero_division=0)

    result = {
        'config_id': config_id,
        'model_type': stack_type,
        'fe_state': 'clean_fe',
        'mode': mode,
        'panel': panel if panel else 'All',
        'best_threshold': best_thr,
        'f1_default': f1_default,
        'n_train': X_train.shape[0],
        'n_test': X_test.shape[0],
        'n_features': X_train.shape[1],
        **metrics,
        'time_seconds': elapsed,
        '_model': model,
        '_y_test': y_test.values,
        '_y_pred_proba': y_pred_proba,
        '_y_pred': y_pred,
    }
    return result

In [5]:
# === STACKING EGITIM DONGUSU ===
STACKING_RESULTS = {}
total = len(STACKING_CONFIGS)

for i, config in enumerate(STACKING_CONFIGS, 1):
    config_id = config['config_id']
    print(f'\n[{i}/{total}] {config_id}')
    print('-' * 50)

    try:
        result = run_stacking_config(config, df_clean_fe)
        STACKING_RESULTS[config_id] = result
        print(f'  F1={result["f1"]:.4f} | AUC-ROC={result["auc_roc"]:.4f} | '
              f'Threshold={result["best_threshold"]:.2f} | {result["time_seconds"]:.1f}s')
    except Exception as e:
        print(f'  HATA: {e}')
        import traceback
        traceback.print_exc()

print(f'\n{"="*60}')
print(f'Tamamlanan stacking: {len(STACKING_RESULTS)}/{total}')


[1/8] stacking_lr_clean_fe_multi_panel
--------------------------------------------------
  Veri: 3429 train, 858 test, 1499 feature
  Stacking (LR meta-learner) egitiliyor...
  Stacking F1: 0.5842 (threshold: 0.38)
  F1=0.5842 | AUC-ROC=0.7489 | Threshold=0.38 | 107.7s

[2/8] stacking_lr_clean_fe_single_General
--------------------------------------------------
  Veri: 2524 train, 632 test, 1495 feature
  Stacking (LR meta-learner) egitiliyor...
  Stacking F1: 0.5575 (threshold: 0.40)
  F1=0.5575 | AUC-ROC=0.7248 | Threshold=0.40 | 86.5s

[3/8] stacking_lr_clean_fe_single_Hereditary_Cancer
--------------------------------------------------
  Veri: 572 train, 143 test, 1495 feature
  Stacking (LR meta-learner) egitiliyor...
  Stacking F1: 0.6400 (threshold: 0.36)
  F1=0.6400 | AUC-ROC=0.8114 | Threshold=0.36 | 31.2s

[4/8] stacking_lr_clean_fe_single_PAH
--------------------------------------------------
  Veri: 259 train, 65 test, 1495 feature
  Stacking (LR meta-learner) egitiliyor.

In [6]:
# Stacking sonuclarini kaydet
stacking_csv_cols = ['config_id', 'model_type', 'fe_state', 'mode', 'panel',
                     'best_threshold', 'f1_default',
                     'n_train', 'n_test', 'n_features',
                     'f1', 'auc_roc', 'auc_pr', 'mcc', 'precision', 'recall',
                     'specificity', 'balanced_accuracy', 'cohens_kappa', 'time_seconds']

stacking_list = [{k: v for k, v in r.items() if not k.startswith('_')}
                 for r in STACKING_RESULTS.values()]
stacking_df = pd.DataFrame(stacking_list)[stacking_csv_cols]

os.makedirs(RESULTS_V2_DIR, exist_ok=True)
stacking_df.to_csv(os.path.join(RESULTS_V2_DIR, 'stacking_results.csv'), index=False)

print('STACKING SONUCLARI:')
print(stacking_df[['config_id', 'f1', 'auc_roc', 'mcc', 'precision', 'recall']].to_string(index=False))

STACKING SONUCLARI:
                                      config_id       f1  auc_roc      mcc  precision   recall
               stacking_lr_clean_fe_multi_panel 0.584242 0.748875 0.339187   0.437387 0.879562
            stacking_lr_clean_fe_single_General 0.557491 0.724837 0.301072   0.422164 0.820513
  stacking_lr_clean_fe_single_Hereditary_Cancer 0.640000 0.811391 0.487763   0.524590 0.820513
                stacking_lr_clean_fe_single_PAH 0.526316 0.602814 0.222976   0.416667 0.714286
             stacking_lgbm_clean_fe_multi_panel 0.578882 0.731642 0.326479   0.438795 0.850365
          stacking_lgbm_clean_fe_single_General 0.541213 0.692965 0.269747   0.388393 0.892308
stacking_lgbm_clean_fe_single_Hereditary_Cancer 0.630631 0.817554 0.482487   0.486111 0.897436
              stacking_lgbm_clean_fe_single_PAH 0.512195 0.508658 0.176908   0.344262 1.000000


In [7]:
# Tek model vs Stacking karsilastirmasi
# 02_training_clean.ipynb sonuclarini yukle
single_model_path = os.path.join(RESULTS_V2_DIR, 'model_comparison_results.csv')

if os.path.exists(single_model_path):
    single_df = pd.read_csv(single_model_path)
    # Sadece clean_fe sonuclarini al
    single_clean_fe = single_df[single_df['fe_state'] == 'clean_fe'].copy()

    print('=' * 70)
    print('TEK MODEL vs STACKING KARSILASTIRMASI (clean_fe)')
    print('=' * 70)

    # Panel bazinda en iyi tek model vs stacking
    comparison_rows = []
    for panel in ['All'] + PANELS_SINGLE:
        # En iyi tek model
        panel_single = single_clean_fe[single_clean_fe['panel'] == panel]
        if len(panel_single) > 0:
            best_single = panel_single.loc[panel_single['f1'].idxmax()]
            best_single_f1 = best_single['f1']
            best_single_model = best_single['model_type']
        else:
            best_single_f1 = None
            best_single_model = '-'

        # En iyi stacking
        panel_stack = stacking_df[stacking_df['panel'] == panel]
        if len(panel_stack) > 0:
            best_stack = panel_stack.loc[panel_stack['f1'].idxmax()]
            best_stack_f1 = best_stack['f1']
            best_stack_model = best_stack['model_type']
        else:
            best_stack_f1 = None
            best_stack_model = '-'

        delta = (best_stack_f1 - best_single_f1) if (best_stack_f1 and best_single_f1) else None

        comparison_rows.append({
            'panel': panel,
            'best_single_model': best_single_model,
            'single_f1': best_single_f1,
            'best_stack_model': best_stack_model,
            'stacking_f1': best_stack_f1,
            'delta_f1': delta,
        })

    comparison_df = pd.DataFrame(comparison_rows)
    comparison_df.to_csv(os.path.join(RESULTS_V2_DIR, 'single_vs_stacking.csv'), index=False)
    print(comparison_df.to_string(index=False))

    # Stacking kazanci ozet
    if comparison_df['delta_f1'].notna().any():
        avg_delta = comparison_df['delta_f1'].mean()
        print(f'\nOrtalama stacking kazanci: {avg_delta:+.4f} F1')
else:
    print('Tek model sonuclari bulunamadi. Once 02_training_clean.ipynb calistirilmali.')

TEK MODEL vs STACKING KARSILASTIRMASI (clean_fe)
            panel best_single_model  single_f1 best_stack_model  stacking_f1  delta_f1
              All           xgboost   0.579602      stacking_lr     0.584242  0.004640
          General          lightgbm   0.612069      stacking_lr     0.557491 -0.054578
Hereditary_Cancer           xgboost   0.652632      stacking_lr     0.640000 -0.012632
              PAH          lightgbm   0.603774      stacking_lr     0.526316 -0.077458

Ortalama stacking kazanci: -0.0350 F1


In [8]:
# En iyi stacking modelleri kaydet
os.makedirs(MODELS_V2_DIR, exist_ok=True)

for panel in ['All'] + PANELS_SINGLE:
    panel_results = {k: v for k, v in STACKING_RESULTS.items()
                     if v['panel'] == panel}
    if not panel_results:
        continue

    best_key = max(panel_results, key=lambda k: panel_results[k]['f1'])
    best = panel_results[best_key]

    model_name = f'best_stacking_{panel}_{best["model_type"]}'
    joblib.dump(best['_model'], os.path.join(MODELS_V2_DIR, f'{model_name}.joblib'))
    print(f'Kaydedildi: {model_name} (F1={best["f1"]:.4f})')

print('\nTum stacking modelleri kaydedildi.')

Kaydedildi: best_stacking_All_stacking_lr (F1=0.5842)
Kaydedildi: best_stacking_General_stacking_lr (F1=0.5575)
Kaydedildi: best_stacking_Hereditary_Cancer_stacking_lr (F1=0.6400)
Kaydedildi: best_stacking_PAH_stacking_lr (F1=0.5263)

Tum stacking modelleri kaydedildi.
